# Labo 27 · Carnet Colab — Ton MLP sur MNIST

Dans la page du labo, ton perceptron multicouche NumPy apprenait sur 1347 chiffres de 8 × 8. Ici, il passe à **60 000** chiffres de 28 × 28 pixels.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

## 1. Les données

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
X_train = x_train.reshape(len(x_train), -1).astype("float32") / 255.0
X_test = x_test.reshape(len(x_test), -1).astype("float32") / 255.0
Y_train = np.eye(10, dtype="float32")[y_train]
print(X_train.shape, Y_train.shape, X_test.shape)

## 2. Ton perceptron multicouche, en NumPy

**À toi** : colle ici ta boucle d'entraînement de l'exercice 27.6, adaptée à 784 entrées et 128 neurones cachés (initialisation de He). Entraîne 5 époques en lots de 64 avec η = 0.1, en mesurant le temps total avec `time.time()`. Affiche l'exactitude au test après chaque époque.

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

rng = np.random.default_rng(0)
W1, b1 = rng.normal(0, np.sqrt(2 / 784), (784, 128)).astype("float32"), np.zeros(128, "float32")
W2, b2 = rng.normal(0, np.sqrt(2 / 128), (128, 10)).astype("float32"), np.zeros(10, "float32")

# À toi : 5 époques, lots de 64, η = 0.1; mesure le temps; affiche l'exactitude au test à chaque époque
temps_numpy = None
exactitude_numpy = None

**Questions.** Combien de paramètres a ton réseau? Quels chiffres confond-il le plus? (Indice : `np.argmax` des prédictions, puis une matrice de confusion avec `sklearn.metrics.confusion_matrix`.)

In [ ]:
# À toi : la matrice de confusion de ton réseau sur le test, et les deux chiffres les plus souvent confondus

## 3. La même architecture avec scikit-learn

`MLPClassifier` fait exactement le même calcul. Donne-lui une couche cachée de 128 neurones ReLU, l'optimiseur `"sgd"` avec `learning_rate_init=0.1`, des lots de 64 et 5 itérations (`max_iter=5`, une itération = une époque). Mesure le temps et l'exactitude.

In [ ]:
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings("ignore")   # scikit-learn avertit qu'il n'a pas convergé en 5 époques

# À toi
temps_sklearn = None
exactitude_sklearn = None

## 4. Avant-goût du cours 28 : Keras

Le même réseau en Keras. La rétropropagation est faite pour toi, et le calcul se fait sur le GPU.

In [ ]:
modele = keras.Sequential([
    keras.Input(shape=(784,)),
    keras.layers.Dense(128, activation="relu", kernel_initializer="he_normal"),
    keras.layers.Dense(10, activation="softmax"),
])
modele.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
debut = time.time()
modele.fit(X_train, y_train, epochs=5, batch_size=64, verbose=2)
temps_keras = time.time() - debut
exactitude_keras = modele.evaluate(X_test, y_test, verbose=0)[1]
print(f"Keras : {exactitude_keras:.4f} en {temps_keras:.1f} s")

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : l'exactitude au test et le temps de tes trois modèles (NumPy, scikit-learn, Keras), la confusion la plus fréquente, et une phrase sur ce qui explique les écarts de temps.